# Prototype Similarity Diagnostic Notebook

1. Content-based similarity (exact code Jaccard, full population)
2. Temporal perturbation battery (identity / adjacent-swap / within-trimester shuffle / full shuffle / reversal)
3. Leave-one-event-out (LOO) attribution, trained-model space, per class
4. Full-population confirmation of LOO-identified drivers (simple presence/absence, no risk-factor categories)
5. Local-vs-global classifier comparison, all metrics (AUROC, AUPRC, F1, Balanced Accuracy, Brier)


In [0]:
%pip install "numpy<2.0.0" --force-reinstall
dbutils.library.restartPython()

In [0]:
# # %pip install torch==2.1.2 femr==0.2.3 datasets==2.15.0 xformers transformers==4.35.2

%pip install femr==0.2.3 xformers transformers==4.35.2
dbutils.library.restartPython()

%pip install --force-reinstall --no-cache-dir torch==2.1.2 femr==0.2.3 datasets==2.15.0 xformers==0.0.23.post1 transformers==4.35.2
dbutils.library.restartPython()

import sys
print([p for p in sys.path if 'site-packages' in p])

%pip install --force-reinstall --no-cache-dir "urllib3<2"
dbutils.library.restartPython()

from huggingface_hub import login

hf_token = dbutils.secrets.get(scope="hf-secrets", key="hf_token")
login(token=hf_token)

## 0. Load export + reconstruct trained adapter/prototype layer

In [0]:
import pickle, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr, chisquare, fisher_exact

EXPORT_PATH = "/path/to/exports/full_analysis_export.pkl"
MEDS_PATH = "/path/to/exports/meds_output_spark/data"

with open(EXPORT_PATH, "rb") as f:
    payload = pickle.load(f)

global_projection_map = payload["global_projection_map"]
local_projection_map = payload["local_projection_map"]
global_similarity_df = payload["global_similarity_df"]
pids_test = payload["pids_test"]
Y_fusion_test = payload["Y_fusion_test"]
CLASS_NAMES = payload["CLASS_NAMES"]
global_X_test = payload["global_X_test"]
global_pid_test = np.array([str(p) for p in payload["global_pid_test"]])
global_proto_indices = payload["global_proto_indices"]
p_test = payload["p_test"]
n_global_prototypes = payload["n_global_prototypes"]
adapter_cfg = payload["global_adapter_cfg"]

pid_to_test_idx = {str(pid): i for i, pid in enumerate(pids_test)}
global_pid_to_row = {str(pid): i for i, pid in enumerate(global_pid_test)}
proto_class = pd.Series({p_idx: info["class_name"] for p_idx, info in global_projection_map.items()})
proto_idx_to_patient_id = {p_idx: str(info["patient_id"]) for p_idx, info in global_projection_map.items()}

print(f"Loaded: {len(global_pid_test)} global test patients, {len(pids_test)} in fusion population, "
     f"{n_global_prototypes} global prototypes")


In [0]:
class AdapterMLP(nn.Module):
    def __init__(self, embedding_dim, adapter_hidden, adapter_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, adapter_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(adapter_hidden, adapter_out),
            nn.BatchNorm1d(adapter_out, affine=False),
        )
    def forward(self, x):
        return self.net(x)


class PrototypeLayer(nn.Module):
    def __init__(self, n_prototypes, adapter_out, n_classes=3):
        super().__init__()
        self.prototypes = nn.Parameter(torch.zeros(n_prototypes, adapter_out))
        self.scale = adapter_out ** 0.5
        self.register_buffer("assignment", torch.zeros(n_classes, n_prototypes))
    def forward(self, z):
        z_norm = torch.nn.functional.normalize(z, dim=1)
        p_norm = torch.nn.functional.normalize(self.prototypes, dim=1)
        return self.scale * (z_norm @ p_norm.T)


global_adapter = AdapterMLP(adapter_cfg["embedding_dim"], adapter_cfg["adapter_hidden"], adapter_cfg["adapter_out"])
global_adapter.load_state_dict(payload["global_adapter_state_dict"])
global_adapter.eval()

global_proto_layer = PrototypeLayer(n_global_prototypes, adapter_cfg["adapter_out"], n_classes=len(CLASS_NAMES))
global_proto_layer.load_state_dict(payload["global_proto_layer_state_dict"])
global_proto_layer.eval()

check = payload["verification_check"]
check_row = global_pid_to_row[str(check["patient_id"])]
with torch.no_grad():
    z = global_adapter(torch.tensor(global_X_test[check_row].reshape(1, -1), dtype=torch.float32))
    sims = global_proto_layer(z).cpu().numpy()[0]
reconstructed_sim = sims[global_proto_indices.index(check["prototype_idx"])]
assert abs(reconstructed_sim - check["expected_similarity"]) < 1e-4, \
    f"RECONSTRUCTION MISMATCH ({reconstructed_sim:.6f} vs {check['expected_similarity']:.6f}) -- STOP."
print(f"Adapter/prototype-layer reconstruction VERIFIED (expected {check['expected_similarity']:.6f}, "
     f"got {reconstructed_sim:.6f})")


def project_to_trained_similarity(raw_embedding):
    with torch.no_grad():
        xb = torch.tensor(raw_embedding.reshape(1, -1), dtype=torch.float32)
        z = global_adapter(xb)
        return global_proto_layer(z).cpu().numpy()[0]


## 1. Nearest-prototype assignment per patient (prerequisite for population-scale confirmation)

Builds `full_confusion_df`: each test patient's true class and the class of their single
nearest prototype. Used below to define PROXIMAL (nearest prototype matches true class) vs.
DISTAL (nearest prototype is a different class) groups for the full-population confirmation
step.

In [0]:
def get_true_class(pid):
    pid = str(pid)
    if pid in pid_to_test_idx:
        row_idx = pid_to_test_idx[pid]
        positive = [CLASS_NAMES[c] for c in range(len(CLASS_NAMES)) if Y_fusion_test[row_idx, c] == 1]
        return positive[0] if positive else "control"
    return None


confusion_rows = []
for pid in global_pid_test:
    true_cls = get_true_class(pid)
    if true_cls is None:
        continue
    sims = global_similarity_df.loc[pid]
    nearest_proto = sims.idxmax()
    confusion_rows.append({
        "patient_id": pid, "true_class": true_cls,
        "matched_prototype_class": proto_class[nearest_proto],
        "similarity": sims[nearest_proto],
    })
full_confusion_df = pd.DataFrame(confusion_rows)
print(f"Built nearest-prototype assignment for {len(full_confusion_df)} patients")


## 2. Content-based similarity: exact code Jaccard, full population

For every (test patient, global prototype) pair, computes the Jaccard index between the
patient's OMOP code set and the prototype's source patient's code set, then correlates this
against the model's own similarity score across all pairs.

In [0]:
OMOP_TABLES = {
    "condition": ("catalog.schema.condition_occurrence_cohort", "condition_concept_id", "condition_source_concept_id"),
    "procedure": ("catalog.schema.procedure_occurrence_cohort", "procedure_concept_id", "procedure_source_concept_id"),  # ASSUMED column names -- verify against real schema
    "drug": ("catalog.schema.drug_exposure_cohort", "drug_concept_id", "drug_concept_id"),
    "measurement": ("catalog.schema.measurement_cohort", "measurement_concept_id", "measurement_source_concept_id"),
}


def fetch_code_sets_batched(person_ids):
    """One query PER DOMAIN, batched over ALL person_ids at once -- not
    one query per patient."""
    person_ids = list(set(person_ids))
    codes_by_patient = {pid: set() for pid in person_ids}
    id_list_sql = ", ".join(f"'{p}'" for p in person_ids)
    for domain, (table, concept_col, source_col) in OMOP_TABLES.items():
        df = spark.sql(f"""
            SELECT person_id, {concept_col} AS concept_id, {source_col} AS source_concept_id
            FROM {table} WHERE person_id IN ({id_list_sql})
        """).toPandas()
        if df.empty:
            continue
        df["person_id"] = df["person_id"].astype(str)
        cid = df["concept_id"].fillna(0).astype(int)
        sid = df["source_concept_id"].fillna(0).astype(int)
        df["resolved_id"] = cid.where(cid != 0, sid)
        df = df[df["resolved_id"] != 0]
        for pid, rid in zip(df["person_id"], df["resolved_id"]):
            if pid in codes_by_patient:
                codes_by_patient[pid].add((domain, int(rid)))
    return codes_by_patient


all_prototype_patient_ids = [info["patient_id"] for info in global_projection_map.values()]
test_patient_code_sets = fetch_code_sets_batched(pids_test)
prototype_code_sets_raw = fetch_code_sets_batched(all_prototype_patient_ids)
global_prototype_code_sets = {p_idx: prototype_code_sets_raw[str(info["patient_id"])]
                              for p_idx, info in global_projection_map.items()}

print(f"Fetched code sets for {len(test_patient_code_sets)} test patients, "
     f"{len(global_prototype_code_sets)} global prototypes")


def raw_jaccard(codes_a, codes_b):
    union = codes_a | codes_b
    return len(codes_a & codes_b) / len(union) if union else np.nan


pooled_rows = []
for pid in pids_test:
    patient_codes = test_patient_code_sets.get(str(pid), set())
    sims = global_similarity_df.loc[pid]
    for p_idx in global_proto_indices:
        proto_codes = global_prototype_code_sets.get(p_idx, set())
        pooled_rows.append({
            "patient_id": pid, "prototype_idx": p_idx, "prototype_class": proto_class[p_idx],
            "similarity": sims[p_idx],
            "raw_jaccard": raw_jaccard(patient_codes, proto_codes),
        })
pooled_df = pd.DataFrame(pooled_rows)

rho, p_val = spearmanr(pooled_df["similarity"], pooled_df["raw_jaccard"], nan_policy="omit")
print(f"\nPooled across {pooled_df['patient_id'].nunique()} patients, {len(pooled_df)} (patient, prototype) pairs")
print(f"Spearman rho (similarity vs. exact-code Jaccard): {rho:.3f} (p={p_val:.2e})")


In [0]:
# ============================================================================
# SECTION B (FIXED): Nearest-prototype confusion matrix + enrichment
# ============================================================================
print("="*70, "\nSECTION B (full population): Nearest-prototype confusion matrix + enrichment\n" + "="*70)

FLAGS_TABLE = "catalog.schema.step7_filtered_cohort_with_apos"  # same source table used throughout this project

def fetch_true_labels_for_population(patient_ids):
    """Pulls true class labels directly from the cohort table for ANY
    set of patient_ids -- independent of the fusion-restricted
    population, unlike Y_fusion_test."""
    id_list_sql = ", ".join(f"'{p}'" for p in patient_ids)
    flags_df = spark.sql(f"""
        SELECT person_id, has_pe, has_inferred_pe, has_ghtn, has_gdm, is_control
        FROM {FLAGS_TABLE} WHERE person_id IN ({id_list_sql})
    """).toPandas()
    flags_df["person_id"] = flags_df["person_id"].astype(str)

    def resolve_class(row):
        if row["is_control"] == 1:
            return "control"
        if row["has_pe"] == 1 or row["has_inferred_pe"] == 1 or row["has_ghtn"] == 1:
            return "pe_ghtn"
        if row["has_gdm"] == 1:
            return "gdm"
        return None  # no positive flag and not control -- shouldn't normally occur, but don't guess

    flags_df["true_class"] = flags_df.apply(resolve_class, axis=1)
    return flags_df.set_index("person_id")["true_class"].to_dict()


true_class_lookup = fetch_true_labels_for_population(global_pid_test.tolist())
n_unresolved = sum(1 for pid in global_pid_test if true_class_lookup.get(str(pid)) is None)
print(f"Fetched true labels for {len(true_class_lookup)} patients "
     f"({n_unresolved} unresolved -- no positive flag and not control, check these manually if > 0)\n")

def get_true_class_full_pop(pid):
    return true_class_lookup.get(str(pid))


confusion_rows = []
for pid in global_pid_test:
    true_cls = get_true_class_full_pop(pid)
    if true_cls is None:
        continue
    sims = global_similarity_df.loc[pid]
    nearest_proto = sims.idxmax()
    confusion_rows.append({
        "patient_id": pid, "true_class": true_cls,
        "matched_prototype_class": proto_class[nearest_proto],
        "similarity": sims[nearest_proto],
    })
full_confusion_df = pd.DataFrame(confusion_rows)

n_dropped = len(global_pid_test) - len(full_confusion_df)
print(f"Built confusion data for {len(full_confusion_df)} / {len(global_pid_test)} patients "
     f"({n_dropped} dropped -- unresolved label)\n")

proto_class_counts = pd.Series([info["class_name"] for info in global_projection_map.values()]).value_counts(normalize=True)
print("True chance level per class (fraction of all prototypes):")
print(proto_class_counts, "\n")

confusion_results = {}
for true_cls in CLASS_NAMES:
    sub = full_confusion_df[full_confusion_df["true_class"] == true_cls]
    if sub.empty:
        continue
    observed_counts = sub["matched_prototype_class"].value_counts().reindex(CLASS_NAMES, fill_value=0)
    observed_props = observed_counts / observed_counts.sum()
    expected_props = proto_class_counts.reindex(CLASS_NAMES, fill_value=1e-9)
    expected_counts = expected_props.values * observed_counts.sum()
    chi2, p_val = chisquare(observed_counts.values, f_exp=expected_counts)

    print(f"True class: {true_cls} (n={len(sub)})")
    for matched_cls in CLASS_NAMES:
        obs, exp = observed_props[matched_cls], expected_props[matched_cls]
        print(f"  -> {matched_cls:10s} observed={obs*100:5.1f}%  chance={exp*100:5.1f}%  enrichment={obs/exp:.2f}x")
    print(f"  Chi-square vs. chance: chi2={chi2:.2f}, p={p_val:.4f}\n")
    confusion_results[true_cls] = {"observed": observed_props, "expected": expected_props, "chi2": chi2, "p": p_val}

In [0]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================================
# Small, clean 3x3 enrichment matrix: rows = true class, columns = matched
# prototype class, cell values = enrichment ratio (observed/chance), with
# large bold text for readability. Reuses confusion_results built above.
# ============================================================================
matrix = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)))
for i, true_cls in enumerate(CLASS_NAMES):
    for j, matched_cls in enumerate(CLASS_NAMES):
        obs = confusion_results[true_cls]["observed"][matched_cls]
        exp = confusion_results[true_cls]["expected"][matched_cls]
        matrix[i, j] = obs / exp

fig, ax = plt.subplots(figsize=(4.2, 4.2))
im = ax.imshow(matrix, cmap="RdBu_r", vmin=0, vmax=2, aspect="equal")

for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        val = matrix[i, j]
        color = "white" if val > 1.6 or val < 0.4 else "black"
        ax.text(j, i, f"{val:.2f}\u00d7", ha="center", va="center",
               fontsize=18, color=color)

labels = {"control": "Control", "pe_ghtn": "PE_GHTN", "gdm": "GDM"}
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels([labels[c] for c in CLASS_NAMES], fontsize=14)
ax.set_yticklabels([labels[c] for c in CLASS_NAMES], fontsize=14)
ax.set_xlabel("Matched Prototype Class", fontsize=16, labelpad=10)
ax.set_ylabel("True Class", fontsize=16, labelpad=10)
ax.set_title("Nearest-Prototype\nEnrichment Over Chance", fontsize=17, fontweight="bold", pad=12)

ax.set_xticks(np.arange(-0.5, len(CLASS_NAMES), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(CLASS_NAMES), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2.5)
ax.tick_params(which="minor", length=0)

plt.tight_layout()
plt.savefig("prototype_enrichment_matrix.png", dpi=300, bbox_inches="tight")
print("saved prototype_enrichment_matrix.png")

## 3. Temporal perturbation battery

Five conditions forming a severity gradient, not just shuffle-vs-reverse: identity (sanity
check -- should be ~1.0), a mild local perturbation (adjacent-pair swap), a
coarse-timing-preserving perturbation (shuffle within trimester, preserving which trimester
each event falls in), full random shuffle, and full reversal. Each is compared against the
patient's OWN original (unperturbed) embedding via cosine similarity.

In [0]:
import femr.models.transformer, femr.models.tokenizer, femr.models.processor
import datetime

model_name = "StanfordShahLab/clmbr-t-base"
tokenizer = femr.models.tokenizer.FEMRTokenizer.from_pretrained(model_name)
batch_processor = femr.models.processor.FEMRBatchProcessor(tokenizer)
model = femr.models.transformer.FEMRModel.from_pretrained(model_name)
model.eval()

BIRTH_CODE = "SNOMED/184099003"


import pyarrow.dataset as ds
import pyarrow.compute as pc
from itertools import groupby
from operator import itemgetter

_meds_ds_cache = {}

def get_patient_meds_events(patient_id, meds_path=MEDS_PATH):
    """Events built exactly as the extraction notebook (clmbr_t_base_modeling) builds them: pyarrow read of
    (subject_id, time, code), sort_values(['subject_id','time']), then group by time. The earlier Spark-read version gave the
    same event counts but not the same event lists (min cosine to the stored embeddings 0.39), so LOO baselines were off."""
    if meds_path not in _meds_ds_cache:
        _meds_ds_cache[meds_path] = ds.dataset(meds_path, format="parquet")
    df = _meds_ds_cache[meds_path].to_table(columns=["subject_id", "time", "code"],
                                            filter=pc.field("subject_id") == int(patient_id)).to_pandas()
    df = df.sort_values(["subject_id", "time"])
    rows = zip(df["time"].to_numpy().astype("datetime64[us]").tolist(), df["code"].to_numpy().tolist())
    return [{"time": t, "measurements": [{"code": c} for _, c in grp]} for t, grp in groupby(rows, key=itemgetter(0))]

def embed_events(patient_id, events):
    """Final-event embedding exactly as the extraction notebook picks it: among the rows the model returns, the one at the
    latest timestamp (first such row if several share it), after a stable sort by time. The earlier version took the raw
    last row, which did not reproduce the stored embeddings (min cosine 0.48, median 0.71 on 20 patients)."""
    example_patient = {"patient_id": int(patient_id), "events": events}
    raw_batch = batch_processor.convert_patient(example_patient, tensor_type="pt")
    batch = batch_processor.collate([raw_batch])
    with torch.no_grad():
        _, result = model(**batch)
    times = result["timestamps"].cpu().numpy()
    reps = result["representations"].cpu().numpy()
    order = np.argsort(times, kind="stable")
    times, reps = times[order], reps[order]
    return reps[np.flatnonzero(times == times.max())[0]]


def is_birth_event(event):
    return any(m["code"] == BIRTH_CODE for m in event["measurements"])


def perturb_identity(events, episode_start=None, seed=0):
    """Sanity check: re-embeds the UNCHANGED sequence. Should give
    cosine ~1.0 -- confirms embed_events is deterministic and that any
    non-1.0 result from other perturbations reflects the perturbation
    itself, not pipeline noise."""
    return list(events)


def perturb_adjacent_swap(events, episode_start=None, seed=0):
    """Mildest real perturbation: swaps each adjacent PAIR of
    (non-birth) events. Preserves both content and coarse position,
    disturbs only fine-grained local order."""
    rng = np.random.RandomState(seed)
    idx = [i for i, e in enumerate(events) if not is_birth_event(e)]
    out = list(events)
    pairs = [(idx[i], idx[i+1]) for i in range(0, len(idx) - 1, 2)]
    for a, b in pairs:
        out[a], out[b] = out[b], out[a]
    return out


def perturb_shuffle_within_trimester(events, episode_start, seed=0):
    rng = np.random.RandomState(seed)
    by_tri = {}
    for e in events:
        if is_birth_event(e):
            by_tri.setdefault("birth", []).append(e)
            continue
        tri = (e["time"] - episode_start).days // 91 if episode_start else 0
        by_tri.setdefault(tri, []).append(e)
    out = []
    for key in sorted(by_tri, key=lambda k: (k != "birth", k if k != "birth" else 0)):
        block = by_tri[key]
        if key == "birth":
            out.extend(block)
            continue
        order = rng.permutation(len(block))
        out.extend([block[i] for i in order])
    return out


def perturb_full_shuffle(events, episode_start=None, seed=0):
    rng = np.random.RandomState(seed)
    removable = [i for i, e in enumerate(events) if not is_birth_event(e)]
    birth = [e for e in events if is_birth_event(e)]
    shuffled = [events[i] for i in rng.permutation(removable)]
    return birth + shuffled


def perturb_reverse(events, episode_start=None, seed=0):
    birth = [e for e in events if is_birth_event(e)]
    rest = [e for e in events if not is_birth_event(e)]
    return birth + rest[::-1]


PERTURBATIONS = {
    "identity": perturb_identity,
    "adjacent_swap": perturb_adjacent_swap,
    "shuffle_within_trimester": perturb_shuffle_within_trimester,
    "full_shuffle": perturb_full_shuffle,
    "reverse": perturb_reverse,
}


def run_temporal_battery(patient_ids, n_seeds=2, meds_path=MEDS_PATH):
    rows = []
    for pid in patient_ids:
        events = get_patient_meds_events(pid, meds_path)
        if len(events) < 4:
            continue
        episode_start = events[0]["time"]
        orig_vec = embed_events(pid, events)
        for name, fn in PERTURBATIONS.items():
            n_runs = n_seeds if name not in ("identity",) else 1
            for seed in range(n_runs):
                perturbed = fn(events, episode_start, seed)
                perturbed_vec = embed_events(pid, perturbed)
                cos = np.dot(orig_vec, perturbed_vec) / (np.linalg.norm(orig_vec) * np.linalg.norm(perturbed_vec) + 1e-9)
                rows.append({"patient_id": pid, "perturbation": name, "seed": seed, "cosine_to_original": cos})
    return pd.DataFrame(rows)


rng = np.random.RandomState(42)
all_patient_ids = pd.unique(pooled_df["patient_id"])
sample_patients_for_temporal = list(rng.choice(all_patient_ids, size=15, replace=False))

battery_df = run_temporal_battery(sample_patients_for_temporal, n_seeds=2)
if not battery_df.empty:
    order = ["identity", "adjacent_swap", "shuffle_within_trimester", "full_shuffle", "reverse"]
    summary = battery_df.groupby("perturbation")["cosine_to_original"].agg(["mean", "std", "count"]).reindex(order)
    print(summary)

In [0]:
# Does the patched loader + embed_events reproduce the stored global embeddings? (aggregates only, ~20 forward passes)
chk = list(np.random.RandomState(0).choice(global_pid_test, 20, replace=False))
cos = []
for p in chk:
    v = embed_events(p, get_patient_meds_events(p)); s = global_X_test[global_pid_to_row[p]]
    cos.append(np.dot(v, s) / (np.linalg.norm(v) * np.linalg.norm(s) + 1e-9))
cos = np.array(cos)
print(f"cosine to stored global embedding: min {cos.min():.5f}, median {np.median(cos):.5f}")
assert cos.min() > 0.999, "loader fix does not reproduce the stored embeddings -- STOP before rerunning LOO"

## 4. Leave-one-out (LOO) attribution, trained-model space, per class

Unchanged from the original diagnostic pipeline: for each of the 15 most confidently-matched
true-positive patients per class, removes one event at a time and measures the drop in
similarity to that patient's single most-similar same-class prototype. Aggregated by code,
with bundling (co-removed codes get identical, inseparable scores) and rarity flags.

In [0]:
def leave_one_event_out_embeddings(patient_id, events):
    removable = [i for i, e in enumerate(events) if not is_birth_event(e)]
    embeddings = {-1: embed_events(patient_id, events)}
    for i in removable:
        embeddings[i] = embed_events(patient_id, events[:i] + events[i+1:])
    return embeddings


def select_best_patients(class_name, n_patients=15):
    class_col = CLASS_NAMES.index(class_name)
    class_proto_indices = [p for p in global_proto_indices if proto_class[p] == class_name]
    true_positive_pids = [pid for pid in pids_test if Y_fusion_test[pid_to_test_idx[str(pid)], class_col] == 1]
    rows = []
    for pid in true_positive_pids:
        sims = global_similarity_df.loc[pid, class_proto_indices]
        best_proto = sims.idxmax()
        rows.append({"patient_id": pid, "prototype_idx": best_proto, "similarity": sims[best_proto]})
    return pd.DataFrame(rows).sort_values("similarity", ascending=False).head(n_patients)


_resolve_cache = {}
def resolve_code_string(code_str):
    if code_str in _resolve_cache:
        return _resolve_cache[code_str]
    if "/" not in code_str:
        return code_str
    vocab, concept_code = code_str.split("/", 1)
    row = spark.sql(f"""
        SELECT concept_name FROM catalog.omop.concept
        WHERE vocabulary_id = '{vocab}' AND concept_code = '{concept_code}' LIMIT 1
    """).collect()
    name = row[0]["concept_name"] if row else code_str
    _resolve_cache[code_str] = name
    return name


def compute_meds_code_prevalence(patient_ids, meds_path=MEDS_PATH):
    id_list_sql = ", ".join(f"'{p}'" for p in patient_ids)
    counts = spark.sql(f"""
        SELECT code, COUNT(DISTINCT subject_id) AS n_patients
        FROM parquet.`{meds_path}` WHERE CAST(subject_id AS STRING) IN ({id_list_sql})
        GROUP BY code
    """).toPandas()
    counts["prevalence"] = counts["n_patients"] / len(patient_ids)
    return counts.set_index("code")["prevalence"]


meds_code_prevalence = compute_meds_code_prevalence(pids_test)


def collect_loo_drivers_with_bundling(class_name, n_patients=15):
    best_patients = select_best_patients(class_name, n_patients=n_patients)
    all_rows = []
    for _, patient_row in best_patients.iterrows():
        pid, proximal_proto_idx = patient_row["patient_id"], patient_row["prototype_idx"]
        events = get_patient_meds_events(pid)
        if len(events) < 2:
            continue
        embeddings_by_variant = leave_one_event_out_embeddings(pid, events)
        if -1 not in embeddings_by_variant:
            continue
        orig_sims = project_to_trained_similarity(embeddings_by_variant[-1])
        orig_prox_sim = orig_sims[global_proto_indices.index(proximal_proto_idx)]
        for i, event in enumerate(events):
            if i not in embeddings_by_variant:
                continue
            perturbed_sims = project_to_trained_similarity(embeddings_by_variant[i])
            prox_sim = perturbed_sims[global_proto_indices.index(proximal_proto_idx)]
            sim_drop = orig_prox_sim - prox_sim
            codes_in_event = [m["code"] for m in event["measurements"]]
            for code in codes_in_event:
                all_rows.append({"patient_id": pid, "code": code, "proximal_sim_drop": sim_drop,
                                "n_codes_in_event": len(codes_in_event)})
    return pd.DataFrame(all_rows)


def aggregate_with_artifact_classification(driver_df, prevalence_lookup, rarity_threshold=0.05):
    if driver_df.empty:
        return pd.DataFrame()
    agg = driver_df.groupby("code").agg(
        mean_proximal_sim_drop=("proximal_sim_drop", "mean"),
        n_patients_with_code=("patient_id", "nunique"),
        n_occurrences=("code", "count"),
        mean_bundle_size=("n_codes_in_event", "mean"),
    ).sort_values("mean_proximal_sim_drop", ascending=False)
    agg["concept_name"] = [resolve_code_string(c) for c in agg.index]
    agg["test_set_prevalence"] = [prevalence_lookup.get(c, np.nan) for c in agg.index]
    def classify(row):
        if row["mean_bundle_size"] > 1.5:
            return "bundled_event"
        elif pd.notna(row["test_set_prevalence"]) and row["test_set_prevalence"] < rarity_threshold:
            return "rare (manual review)"
        return "clean"
    agg["artifact_type"] = agg.apply(classify, axis=1)
    return agg


loo_results = {}
for class_name in ["pe_ghtn", "gdm"]:
    print(f"\n--- {class_name.upper()} ---")
    driver_df = collect_loo_drivers_with_bundling(class_name, n_patients=15)
    agg = aggregate_with_artifact_classification(driver_df, meds_code_prevalence)
    loo_results[class_name] = agg
    if not agg.empty:
        top = agg.head(20)
        print(top["artifact_type"].value_counts())
        print(top[["concept_name", "mean_proximal_sim_drop", "test_set_prevalence", "artifact_type"]].to_string())


## 5. Full-population confirmation of LOO-identified drivers

For every code LOO flagged as a top driver (any artifact type -- clean, bundled, or rare),
tests whether it occurs at a different rate in the FULL population's proximal vs. distal
groups (simple presence/absence, Fisher's exact test).

In [0]:
def check_code_prevalence_proximal_vs_distal(code, class_name):
    """Reverse-maps a FEMR/MEDS code string back to an OMOP resolved_id
    so it can be checked against test_patient_code_sets, then compares
    presence/absence between the FULL proximal and distal groups."""
    if "/" not in code:
        return None
    vocab, concept_code = code.split("/", 1)
    row = spark.sql(f"""
        SELECT concept_id FROM catalog.omop.concept
        WHERE vocabulary_id = '{vocab}' AND concept_code = '{concept_code}' LIMIT 1
    """).collect()
    if not row:
        return None
    resolved_id = int(row[0]["concept_id"])

    sub = full_confusion_df[full_confusion_df["true_class"] == class_name]
    proximal_pids = sub[sub["matched_prototype_class"] == class_name]["patient_id"].tolist()
    distal_pids = sub[sub["matched_prototype_class"] != class_name]["patient_id"].tolist()

    def has_code(pid):
        codes = test_patient_code_sets.get(str(pid), set())
        return any(rid == resolved_id for _, rid in codes)

    prox_has = sum(has_code(p) for p in proximal_pids)
    prox_not = len(proximal_pids) - prox_has
    dist_has = sum(has_code(p) for p in distal_pids)
    dist_not = len(distal_pids) - dist_has

    _, p_val = fisher_exact([[prox_has, prox_not], [dist_has, dist_not]])
    return {
        "code": code, "proximal_rate": prox_has / len(proximal_pids) if proximal_pids else np.nan,
        "distal_rate": dist_has / len(distal_pids) if distal_pids else np.nan,
        "difference": (prox_has/len(proximal_pids) - dist_has/len(distal_pids))
                     if proximal_pids and distal_pids else np.nan,
        "n_proximal": len(proximal_pids), "n_distal": len(distal_pids), "p_value": p_val,
    }


def check_all_loo_drivers(class_name, top_n=30, alpha=0.05):
    """Applies a plain BONFERRONI correction across all codes tested for
    this class (not Holm-Bonferroni) -- alpha_corrected = alpha / n_tests,
    and p_bonferroni = min(1.0, p_raw * n_tests). Simple, conservative,
    and easy to justify to a reviewer given the open-ended (not
    pre-registered) list of candidate codes LOO surfaced."""
    agg = loo_results[class_name]
    if agg.empty:
        print(f"No LOO drivers to check for {class_name}")
        return None
    top = agg.head(top_n)
    rows = []
    for code, row in top.iterrows():
        result = check_code_prevalence_proximal_vs_distal(code, class_name)
        if result is None:
            continue
        result["artifact_type"] = row["artifact_type"]
        result["concept_name"] = row["concept_name"]
        rows.append(result)
    result_df = pd.DataFrame(rows).sort_values("p_value")

    n_tests = len(result_df)
    result_df["p_bonferroni"] = (result_df["p_value"] * n_tests).clip(upper=1.0)
    result_df["significant_bonferroni"] = result_df["p_bonferroni"] < alpha

    n_significant_raw = (result_df["p_value"] < alpha).sum()
    n_significant_corrected = result_df["significant_bonferroni"].sum()
    print(f"{n_tests} codes tested. Significant at raw p<{alpha}: {n_significant_raw}. "
         f"Significant after Bonferroni correction (alpha_corrected={alpha/n_tests:.2e}): {n_significant_corrected}\n")
    print(result_df[["concept_name", "proximal_rate", "distal_rate", "difference",
                     "p_value", "p_bonferroni", "significant_bonferroni", "artifact_type"]].to_string())
    return result_df


population_confirmation = {}
for class_name in ["pe_ghtn", "gdm"]:
    print(f"\n{'='*70}\n{class_name.upper()}: full-population confirmation of LOO drivers\n{'='*70}")
    population_confirmation[class_name] = check_all_loo_drivers(class_name, top_n=30)

import pickle
with open("/path/to/exports/loo_results_v2.pkl", "wb") as f:
    pickle.dump({"loo_results": loo_results, "population_confirmation": population_confirmation}, f)


In [0]:
# ============================================================================
# CELL 1 -- Rebuild the frozen fusion pipeline from the export payload.
# Depends on earlier notebook cells: payload, AdapterMLP, PrototypeLayer, global_adapter,
# global_proto_layer, global_proto_indices, n_global_prototypes, CLASS_NAMES, pids_test,
# p_test, global_pid_to_row, global_X_test.
#
# Design: NO retraining. The payload already holds both frozen branches and the fusion weights,
# i.e. the exact model the LOO drivers were found under. The only things it does not hold are the
# z-score / age statistics (fit on the TRAIN split), so those are recomputed here from the
# training embeddings, then checked against the stored p_test.
# ============================================================================
import numpy as np
import pandas as pd
import torch
import pyspark.sql.functions as sf
from sklearn.model_selection import train_test_split

CFG_FUSION_CHECK = {
    # ---- what to evaluate ----
    "target_class": "gdm",
    "n_patients": 15,                  # same patients as LOO (select_best_patients top-N by similarity)
    "modes": ["event", "code"],        # "event": remove whole event(s) containing the code (as LOO did)
                                       # "code":  remove only that code's measurement, keep co-occurring codes
    # ---- random-removal control ----
    "n_random": 20,                    # random removals drawn per driver removal (same number of units)
    "n_match": 5,                      # nearest-cosine random removals used as the matched control
    "seed": 42,
    # ---- data (same sources/seed as the fusion notebook's full-pregnancy run) ----
    "flags_path": "catalog.schema.step7_filtered_cohort_with_apos",
    "split_seed": 42,                  # = random_seed in FINAL_CFG_GLOBAL / FINAL_CFG_LOCAL
    "global_emb_table": "catalog.schema.patient_clmbr_embeddings_new_v5",
    # ASSUMED table name (taken from the commented line in 02_landmark_train_eval) -- verify it is the
    # local table used for the fusion notebook's LOCAL branch:
    "local_emb_table": "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_trimester_delta",
    "topk_pool_k": 5,                  # = topk_pool_k in FINAL_CFG_LOCAL
    "chunk": 8192,                     # rows per adapter forward pass
    "verify_tol": 1e-3,                # max |recomputed fused prob - stored p_test|
}
C = CFG_FUSION_CHECK
FUSION_DEVICE = "cpu"   # adapters are tiny MLPs; keeps this independent of the CLMBR model's device


# ---- frozen LOCAL branch from payload (global branch already reconstructed in the earlier cell) ----
lcfg = payload["local_adapter_cfg"]
n_local_prototypes = payload["n_local_prototypes"]
local_adapter = AdapterMLP(lcfg["embedding_dim"], lcfg["adapter_hidden"], lcfg["adapter_out"])
local_adapter.load_state_dict(payload["local_adapter_state_dict"]); local_adapter.eval()
local_proto_layer = PrototypeLayer(n_local_prototypes, lcfg["adapter_out"], n_classes=len(CLASS_NAMES))
local_proto_layer.load_state_dict(payload["local_proto_layer_state_dict"]); local_proto_layer.eval()

# Fusion input columns are in prototype-index order 0..n-1; the payload's similarity DataFrames are keyed
# by sorted prototype index, so this must hold for column alignment to be valid.
assert list(global_proto_indices) == list(range(n_global_prototypes)), "global prototype indices not contiguous"
assert sorted(payload["local_proto_indices"]) == list(range(n_local_prototypes)), "local prototype indices not contiguous"

FUSION_W = payload["fusion_classifier_weight"].astype(np.float32)   # (n_classes, n_local + n_global + 1)
FUSION_B = payload["fusion_classifier_bias"].astype(np.float32)
assert FUSION_W.shape[1] == n_local_prototypes + n_global_prototypes + 1


def sims_from_embeddings(adapter, proto_layer, X, chunk=C["chunk"]):
    """Raw prototype similarities (scale * cosine) for a matrix of 768-d embeddings, chunked."""
    out = []
    with torch.no_grad():
        for s in range(0, len(X), chunk):
            xb = torch.as_tensor(X[s:s + chunk], dtype=torch.float32, device=FUSION_DEVICE)
            out.append(proto_layer(adapter(xb)).cpu().numpy())
    return np.concatenate(out, axis=0)


def pool_topk_by_patient(sims, pids, k):
    """Vectorized local-branch pooling: per patient, mean of the top-k row similarities per prototype
    (k_eff = min(k, n_rows)). Same result as the export cell's per-patient np.partition loop."""
    codes, uniques = pd.factorize(pids)
    order = np.argsort(codes, kind="stable")
    codes_s, sims_s = codes[order], sims[order]
    counts = np.bincount(codes_s, minlength=len(uniques))
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
    pos = np.arange(len(codes_s)) - starts[codes_s]                 # position of each row within its patient
    padded = np.full((len(uniques), counts.max(), sims.shape[1]), -np.inf, dtype=np.float32)
    padded[codes_s, pos] = sims_s
    padded.sort(axis=1)                                             # ascending; -inf padding sorts to the front
    top = padded[:, -min(k, counts.max()):, :]                      # largest entries sit at the end
    top = np.where(np.isinf(top), 0.0, top)                         # padding contributes 0 to the sum
    return np.asarray(uniques), (top.sum(axis=1) / np.minimum(counts, k)[:, None]).astype(np.float32)


def zscore_fit(X):
    return X.mean(axis=0, keepdims=True), X.std(axis=0, keepdims=True).clip(min=1e-8)   # same as the fusion notebook


def _build_person_split(flags_path, seed):
    """Copy of the fusion notebook's build_person_split (80/10/10, stratified on the label combo), so the
    TRAIN set -- and hence the z-score stats -- is identical to the one the fusion layer was trained with."""
    flags_df = spark.table(flags_path).select(
        "person_id", "date_of_birth", "recorded_episode_start", "inferred_episode_start",
        "has_pe", "has_inferred_pe", "has_ghtn", "has_gdm", "is_control").toPandas()
    flags_df["person_id"] = flags_df["person_id"].astype(str)
    flags_df["y_control"] = flags_df["is_control"].astype(int)
    flags_df["y_pe_ghtn"] = ((flags_df["has_pe"] == 1) | (flags_df["has_inferred_pe"] == 1) | (flags_df["has_ghtn"] == 1)).astype(int)
    flags_df["y_gdm"] = flags_df["has_gdm"].astype(int)
    for c in ["date_of_birth", "recorded_episode_start", "inferred_episode_start"]:
        flags_df[c] = pd.to_datetime(flags_df[c])
    flags_df["episode_start"] = flags_df[["recorded_episode_start", "inferred_episode_start"]].min(axis=1, skipna=True)
    flags_df["age"] = (flags_df["episode_start"] - flags_df["date_of_birth"]).dt.days / 365.25
    combo = (flags_df["y_control"] * 4 + flags_df["y_pe_ghtn"] * 2 + flags_df["y_gdm"]).astype(int)
    pid_tv, pid_test_ = train_test_split(flags_df["person_id"].values, test_size=0.10, stratify=combo, random_state=seed)
    combo_tv = combo[flags_df["person_id"].isin(pid_tv)].values
    pid_train, _ = train_test_split(pid_tv, test_size=0.10 / 0.90, stratify=combo_tv, random_state=seed)
    return set(pid_train), flags_df.set_index("person_id")["age"]


def load_train_inputs():
    """Train-split embeddings for both branches (one Spark join each, no per-patient queries)."""
    train_pids, age_by_pid = _build_person_split(C["flags_path"], C["split_seed"])
    ids = spark.createDataFrame(pd.DataFrame({"pid_str": sorted(train_pids)}))

    def pull(table, local):
        sdf = spark.table(table).withColumn("pid_str", sf.col("patient_id").cast("string"))
        if local:
            sdf = sdf.filter(sf.col("unavailable") == 0)             # same availability filter as load_local_data
        df = sdf.join(sf.broadcast(ids), "pid_str").toPandas()
        emb_cols = sorted([c for c in df.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
        X = df[emb_cols].to_numpy(dtype=np.float32)
        keep = ~np.isnan(X).any(axis=1)                              # drop delta rows that are not yet computable
        return df["pid_str"].to_numpy()[keep], X[keep]

    g_pids, g_X = pull(C["global_emb_table"], local=False)
    l_pids, l_X = pull(C["local_emb_table"], local=True)
    assert len(set(g_pids)) == len(g_pids), "global table has >1 row per patient"
    return g_pids, g_X, l_pids, l_X, age_by_pid


# ---- fit the fusion normalization on the TRAIN split (fusion population = patients present in both branches) ----
g_pids_tr, g_X_tr, l_pids_tr, l_X_tr, age_by_pid = load_train_inputs()
g_sim_tr = sims_from_embeddings(global_adapter, global_proto_layer, g_X_tr)
l_pids_u, l_sim_tr = pool_topk_by_patient(sims_from_embeddings(local_adapter, local_proto_layer, l_X_tr), l_pids_tr, C["topk_pool_k"])
common = np.intersect1d(g_pids_tr, l_pids_u)
g_mean, g_std = zscore_fit(g_sim_tr[pd.Index(g_pids_tr).get_indexer(common)])
l_mean, l_std = zscore_fit(l_sim_tr[pd.Index(l_pids_u).get_indexer(common)])
age_tr = age_by_pid.loc[common].to_numpy(dtype=np.float32)
age_mean, age_std = age_tr.mean(), age_tr.std()
print(f"normalization fit on {len(common)} train patients present in both branches")


def fused_probs(l_sim, g_sim, age):
    """Frozen fusion layer: z-score both similarity vectors + standardized age -> linear -> sigmoid."""
    x = np.concatenate([(l_sim - l_mean) / l_std, (g_sim - g_mean) / g_std,
                        ((age - age_mean) / (age_std + 1e-8)).reshape(-1, 1)], axis=1).astype(np.float32)
    return 1.0 / (1.0 + np.exp(-(x @ FUSION_W.T + FUSION_B)))


# ---- VERIFY: recomputing fused probs from the stored test similarities must reproduce stored p_test ----
pids_test_str = [str(p) for p in pids_test]
p_chk = fused_probs(payload["local_similarity_df"].loc[pids_test_str].to_numpy(),
                    payload["global_similarity_df"].loc[pids_test_str].to_numpy(),
                    np.asarray(payload["age_test"], dtype=np.float32))
err = float(np.abs(p_chk - np.asarray(p_test)).max())
assert err < C["verify_tol"], f"FUSION RECONSTRUCTION MISMATCH (max |dp| = {err:.2e}) -- STOP; check table names / split seed / pooling k."
print(f"Fusion reconstruction VERIFIED: max |recomputed - stored p_test| = {err:.2e} over {len(pids_test_str)} test patients")

In [0]:
# ============================================================================
# CELL 2 -- Remove LOO-identified GDM drivers, re-embed, run BOTH branches + the frozen fusion layer.
# Depends on: cell 1, plus (from the diagnostic notebook) model, batch_processor, get_patient_meds_events,
# is_birth_event, select_best_patients, loo_results, population_confirmation, global_X_test,
# global_pid_to_row, p_test.
#
# One CLMBR forward pass per variant gives a representation at every event time. From that single pass:
#   GLOBAL branch input = representation at the last event (as in the new_v5 extraction)
#   LOCAL  branch input = rolling-trimester window embeddings (representation at the last event <= window end),
#                         then the delta rule (n_back=3), then adapter -> prototype sims -> top-k pooling
# This mirrors clmbr_t_base_modeling (cell 8) and calculate_delta_embeddings, and is GATED below: the
# unmodified patients must reproduce the stored delta-table rows and pooled local similarities first.
# ============================================================================
import copy
C.update({
    # which device runs the CLMBR forward passes. "cpu" = same path as the LOO cell. "cuda" = the device the extraction ran on
    # (use it to test whether CPU-vs-GPU attention explains a gate mismatch; runs a COPY of the model, so the notebook's own
    # CPU model, which other cells use, is untouched)
    "embed_device": "cpu",
    # ---- local branch window grid + delta rule (same values as the extraction / delta notebooks) ----
    "tri_pre_months": 6, "tri_post_months": 13, "tri_stride_months": 1,      # offsets -6..+12 -> 19 windows
    "tri_window_months": 3, "tri_n_back": 3,
    # ---- reproduction gate tolerances (unmodified patients vs stored tables) ----
    "verify_window_rel_tol": 0.05,   # per-row ||recomputed - stored|| / median stored row norm. This runs on CPU without the
                                     # xformers kernel the extraction used, so tiny numeric differences are expected; a
                                     # window-set mismatch or an error far above this means a logic mismatch instead.
    "verify_local_sim_tol": 0.1,     # max |recomputed pooled local sim - stored local_similarity_df| (sims = 8 * cosine)
})
tgt = CLASS_NAMES.index(C["target_class"])

# ---- patients, driver codes, driver ordering ----
pts = select_best_patients(C["target_class"], n_patients=C["n_patients"])["patient_id"].astype(str).tolist()
sig_df = population_confirmation[C["target_class"]]
sig_set = set(sig_df.loc[sig_df["significant_bonferroni"], "code"])
assert sig_set, "no Bonferroni-significant codes found for this class"
# cumulative order = LOO ranking (mean drop in similarity to the proximal prototype, largest first)
ranked_codes = [c for c in loo_results[C["target_class"]]["mean_proximal_sim_drop"].sort_values(ascending=False).index if c in sig_set]

test_row = {p: i for i, p in enumerate(pids_test_str)}
assert all(p in test_row for p in pts), "a selected patient is missing from the fusion test population"
age_pt = np.asarray(payload["age_test"], dtype=np.float32)[[test_row[p] for p in pts]]

# ---- events built EXACTLY as the extraction notebook builds them (pyarrow read, sort_values(['subject_id','time']),
# group by time). The diagnostic notebook's get_patient_meds_events (Spark read + single-column sort) gives the same event
# counts but not the same event lists, and the re-embedded global embeddings then miss the stored ones (min cosine 0.39). ----
import pyarrow.dataset as ds
import pyarrow.compute as pc
from itertools import groupby
from operator import itemgetter
_meds_ds = ds.dataset(MEDS_PATH, format="parquet")

def events_like_extraction(pid):
    df = _meds_ds.to_table(columns=["subject_id", "time", "code"], filter=pc.field("subject_id") == int(pid)).to_pandas()
    df = df.sort_values(["subject_id", "time"])
    rows = zip(df["time"].to_numpy().astype("datetime64[us]").tolist(), df["code"].to_numpy().tolist())
    return [{"time": t, "measurements": [{"code": c} for _, c in grp]} for t, grp in groupby(rows, key=itemgetter(0))]

events_by_pt = [events_like_extraction(p) for p in pts]

# ---- window anchors: episode_start = earlier of recorded/inferred start (same rule as the extraction notebook) ----
fl = (spark.table(C["flags_path"]).select("person_id", "recorded_episode_start", "inferred_episode_start")
      .filter(sf.col("person_id").cast("string").isin(pts)).toPandas())
fl["person_id"] = fl["person_id"].astype(str)
fl["episode_start"] = pd.concat([pd.to_datetime(fl["recorded_episode_start"]), pd.to_datetime(fl["inferred_episode_start"])], axis=1).min(axis=1, skipna=True)
episode_start = fl.set_index("person_id")["episode_start"]
assert all(p in episode_start.index and pd.notna(episode_start[p]) for p in pts), "missing episode_start for a selected patient"

TRI_OFFSETS = list(range(-C["tri_pre_months"], C["tri_post_months"], C["tri_stride_months"]))
TRI_NAMES = [f"tri_mo{o:+03d}" for o in TRI_OFFSETS]

def tri_window_ends(ep):
    # window start = anchor + o months, THEN +window months (not anchor + (o+3) months) -- identical to the extraction,
    # which matters for month-end dates; truncated to seconds like the extraction's astype(times.dtype)
    ends = [(ep + pd.DateOffset(months=o)) + pd.DateOffset(months=C["tri_window_months"]) for o in TRI_OFFSETS]
    return pd.DatetimeIndex(ends).values.astype("datetime64[s]")

W_ENDS = np.stack([tri_window_ends(episode_start[p]) for p in pts])               # (n_patients, n_windows)


# ---- one forward pass -> per-event (time, representation), sorted by time (stable, as in the extraction) ----
embed_model = model if C["embed_device"] == "cpu" else copy.deepcopy(model).to(C["embed_device"])

def _to_dev(obj):                                   # same recursion as move_to_device in the extraction notebook
    if isinstance(obj, torch.Tensor):
        return obj.to(C["embed_device"])
    if isinstance(obj, dict):
        return {k: _to_dev(v) for k, v in obj.items()}
    return obj

def embed_all(pid, events):
    raw_batch = batch_processor.convert_patient({"patient_id": int(pid), "events": events}, tensor_type="pt")
    batch = _to_dev(batch_processor.collate([raw_batch]))
    with torch.no_grad():
        _, result = embed_model(**batch)
    times = result["timestamps"].cpu().numpy().astype("datetime64[s]")
    reps = result["representations"].cpu().numpy()
    order = np.argsort(times, kind="stable")
    return times[order], reps[order]


def extract_windows(times, reps, wends):
    """Same logic as extract_windows_vectorized: a window's embedding is the representation at the last event with
    time <= window end; a window is unavailable if it ends after the last event (data cutoff) or before the first."""
    wends = wends.astype(times.dtype)
    available = ~(wends > times[-1]) & ~(wends < times[0])
    idx = np.clip(np.searchsorted(times, wends, side="right") - 1, 0, len(times) - 1)
    emb = reps[idx].astype(np.float32, copy=True)
    emb[~available] = np.nan
    return emb, available


def embed_variant(pi, events):
    times, reps = embed_all(pts[pi], events)
    g = reps[np.flatnonzero(times == times.max())[0]]                              # last event (first of any ties), as in new_v5
    w, av = extract_windows(times, reps, W_ENDS[pi])
    return g, w, av


def delta_rows(emb, avail, n_back):
    """Vectorized delta rule from calculate_delta_embeddings (no ramp-up): window i -> emb[i] - emb[i-n_back]; the first
    available window keeps its raw cumulative embedding; the n_back-1 windows after it stay NaN (no full-length delta yet)."""
    out = np.full_like(emb, np.nan, dtype=np.float32)
    out[:, n_back:, :] = emb[:, n_back:, :] - emb[:, :-n_back, :]
    has_any = avail.any(axis=1)
    first = np.argmax(avail, axis=1)
    v = np.where(has_any)[0]
    out[v, first[v], :] = emb[v, first[v], :]
    return out


def local_pooled(W_emb, AV):
    """Windows -> delta rows -> (rows with any NaN dropped, as load_local_data does) -> local adapter + prototype layer
    -> top-k pooling per variant. Returns (pooled sims (V, n_local) with NaN if a variant has no rows, delta rows, row mask)."""
    D = delta_rows(W_emb, AV, C["tri_n_back"])
    row_mask = np.isfinite(D).all(axis=2)
    v_idx, w_idx = np.nonzero(row_mask)
    sims = sims_from_embeddings(local_adapter, local_proto_layer, D[v_idx, w_idx])
    uniq, pooled = pool_topk_by_patient(sims, v_idx, C["topk_pool_k"])
    out = np.full((len(W_emb), n_local_prototypes), np.nan, dtype=np.float32)
    out[uniq] = pooled
    return out, D, row_mask


# ---- baselines: re-embed the UNMODIFIED sequences (the reference for every delta, as in LOO) ----
base = [embed_variant(pi, ev) for pi, ev in enumerate(events_by_pt)]
E_base = np.stack([b[0] for b in base]); W_base = np.stack([b[1] for b in base]); AV_base = np.stack([b[2] for b in base])
l_base, D_base, mask_base = local_pooled(W_base, AV_base)

# ---- GATE: unmodified patients must reproduce the stored pipeline before any removal result is trusted ----
# All diagnostics print BEFORE any assert so a failure still shows where the mismatch is.
unit = lambda A: A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)
stored = np.stack([global_X_test[global_pid_to_row[p]] for p in pts])
cos_glob = (unit(E_base) * unit(stored)).sum(1)                                     # (1) forward pass only: last-event embedding vs stored global embedding
print(f"embed device: {C['embed_device']} | global embedding vs stored: cosine min {cos_glob.min():.5f}, median {np.median(cos_glob):.5f}")

emb_cols = [f"emb_{d}" for d in range(E_base.shape[1])]
st = (spark.table(C["local_emb_table"]).filter(sf.col("patient_id").cast("string").isin(pts))
      .filter(sf.col("unavailable") == 0).toPandas())
st["patient_id"] = st["patient_id"].astype(str)
st = st[~st[emb_cols].isna().any(axis=1)]                                           # the same rows load_local_data keeps
v_idx, w_idx = np.nonzero(mask_base)
mine_keys = pd.MultiIndex.from_arrays([[pts[i] for i in v_idx], [TRI_NAMES[w] for w in w_idx]])
st_idx = pd.MultiIndex.from_frame(st[["patient_id", "window_name"]])
assert set(mine_keys) == set(st_idx), (f"WINDOW SET MISMATCH: {len(set(mine_keys) - set(st_idx))} recomputed-only, "
                                       f"{len(set(st_idx) - set(mine_keys))} stored-only windows -- STOP; anchor/grid/availability rule differs.")
ref = st.set_index(["patient_id", "window_name"]).loc[mine_keys, emb_cols].to_numpy(dtype=np.float32)
mine = D_base[v_idx, w_idx]
err = np.linalg.norm(mine - ref, axis=1)
rel = err / np.median(np.linalg.norm(ref, axis=1))                                  # error vs the typical stored row norm
rel_cum = err / np.linalg.norm(W_base[v_idx, w_idx], axis=1)                        # error vs the cumulative embedding behind each row
ref_norm = np.linalg.norm(ref, axis=1)
cos_rows = np.where(ref_norm > 1e-6, (unit(mine) * unit(ref)).sum(1), np.nan)       # exactly-zero stored rows (empty windows) have no direction
is_base = w_idx == np.argmax(AV_base, axis=1)[v_idx]                                # first available window = raw cumulative row, no subtraction
sim_err = float(np.abs(l_base - payload["local_similarity_df"].loc[pts].to_numpy()).max())
for name, m in [("base rows (raw cumulative; forward pass only)", is_base), ("delta rows (emb[i] - emb[i-3])", ~is_base)]:
    print(f"  {name}: n={int(m.sum())} ({int((ref_norm[m] <= 1e-6).sum())} zero rows) | cosine to stored: min {np.nanmin(cos_rows[m]):.4f}, median {np.nanmedian(cos_rows[m]):.4f} "
          f"| err/median stored row norm: median {np.median(rel[m]):.3f}, max {rel[m].max():.3f} | err/cumulative norm: median {np.median(rel_cum[m]):.4f}")
print(f"  max |pooled local sim - stored| = {sim_err:.3e}")
assert rel.max() < C["verify_window_rel_tol"], "delta rows differ from the stored table beyond tolerance -- STOP; read the diagnostics above."
assert sim_err < C["verify_local_sim_tol"], "pooled local similarities differ from stored beyond tolerance -- STOP; read the diagnostics above."


# ---- removal "units": whole events (mode="event") or single measurements (mode="code") ----
def units_for_code(events, code, mode):
    if mode == "event":
        return [(i,) for i, e in enumerate(events) if not is_birth_event(e) and any(m["code"] == code for m in e["measurements"])]
    return [(i, j) for i, e in enumerate(events) if not is_birth_event(e)                # birth event is never removed (as in LOO)
            for j, m in enumerate(e["measurements"]) if m["code"] == code]

def apply_removal(events, units):
    drop_ev = {u[0] for u in units if len(u) == 1}
    drop_ms = {u for u in units if len(u) == 2}
    out = []
    for i, e in enumerate(events):
        if i in drop_ev:
            continue
        ms = [m for j, m in enumerate(e["measurements"]) if (i, j) not in drop_ms]
        if ms:                                                                      # an event with no codes left disappears
            out.append({"time": e["time"], "measurements": ms})
    return out


# ---- enumerate every driver removal; identical removals (e.g. single #1 == cumulative k=1) are embedded once ----
requests, key2idx = [], {}
def request(pi, mode, units):
    key = (pi, mode, frozenset(units))
    if key not in key2idx:
        key2idx[key] = len(requests); requests.append(key)
    return key2idx[key]

specs, n_skipped = [], 0
for pi, events in enumerate(events_by_pt):
    present = [c for c in ranked_codes if any(m["code"] == c for e in events if not is_birth_event(e) for m in e["measurements"])]
    for mode in C["modes"]:
        per_code = {c: set(units_for_code(events, c, mode)) for c in present}
        variants = [("single", r + 1, per_code[c]) for r, c in enumerate(present)]
        variants += [("cumulative", k, set().union(*[per_code[c] for c in present[:k]])) for k in range(1, len(present) + 1)]
        for kind, label, units in variants:
            if not units or not apply_removal(events, units):                   # nothing to remove, or nothing would be left
                n_skipped += 1; continue
            specs.append({"pi": pi, "mode": mode, "kind": kind, "label": label, "n_units": len(units),
                          "emb_idx": request(pi, mode, units)})
print(f"{len(specs)} driver removals ({n_skipped} skipped) -> {len(requests)} unique re-embeddings "
      f"(+{len(pts)} baselines). Each is one CLMBR forward pass (serves both branches).")

# ---- embed (the only slow step) ----
G_req = np.empty((len(requests), E_base.shape[1]), dtype=np.float32)
W_req = np.empty((len(requests),) + W_base.shape[1:], dtype=np.float32)
AV_req = np.empty((len(requests), W_base.shape[1]), dtype=bool)
for n, (pi, mode, units) in enumerate(requests):
    G_req[n], W_req[n], AV_req[n] = embed_variant(pi, apply_removal(events_by_pt[pi], units))
    if (n + 1) % 200 == 0:
        print(f"  embedded {n + 1}/{len(requests)}")

# ---- everything after embedding is batched: both branches + fusion in one pass each ----
pi_req = np.array([k[0] for k in requests])
g_base = sims_from_embeddings(global_adapter, global_proto_layer, E_base)
g_req = sims_from_embeddings(global_adapter, global_proto_layer, G_req)
l_req, _, _ = local_pooled(W_req, AV_req)
P_base = fused_probs(l_base, g_base, age_pt)
ok = ~np.isnan(l_req).any(axis=1)                                                  # a variant with no surviving local rows has no fusion input
P_req = np.full((len(requests), len(CLASS_NAMES)), np.nan, dtype=np.float32)
P_req[ok] = fused_probs(l_req[ok], g_req[ok], age_pt[pi_req[ok]])
cos_req = (unit(G_req) * unit(E_base)[pi_req]).sum(axis=1)                         # cosine(modified, original) of the 768-d global embedding

# sanity: both branches re-embedded should reproduce the stored fused prediction
print(f"max |p_fused(both branches re-embedded) - stored p_test| = {np.abs(P_base - np.asarray(p_test)[[test_row[p] for p in pts]]).max():.4f}")

# ---- assemble results (patients are indexed 0..N-1; raw ids are not printed) ----
keep = [s for s in specs if ok[s["emb_idx"]]]
print(f"{len(specs) - len(keep)} removals dropped: no local window rows left after removal")
pi_v = np.array([s["pi"] for s in keep]); emb_v = np.array([s["emb_idx"] for s in keep])
base_arg = P_base.argmax(axis=1)
variant_df = pd.DataFrame({
    "patient_idx": pi_v, "mode": [s["mode"] for s in keep], "kind": [s["kind"] for s in keep],
    "label": [s["label"] for s in keep], "n_units": [s["n_units"] for s in keep],
    "cos": cos_req[emb_v],
    "delta_true": P_req[emb_v, tgt] - P_base[pi_v, tgt],                           # change in predicted prob of the true class
    "flip": P_req[emb_v].argmax(axis=1) != base_arg[pi_v],                         # argmax class differs from the baseline argmax
    # removing late events moves the patient's data cutoff, so some windows can become unavailable (dropped local rows);
    # this follows the extraction rule literally and is flagged so it can be separated from representation changes
    "avail_changed": (AV_req[emb_v] != AV_base[pi_v]).any(axis=1),
})
print(f"variant_df: {len(variant_df)} rows | window availability changed in {variant_df['avail_changed'].mean():.1%} | "
      f"baseline p({C['target_class']}) over patients: median {np.median(P_base[:, tgt]):.3f}")

In [0]:
# ============================================================================
# CELL 3 -- Descriptive summary: how much do the LOO driver removals change the final (fused) prediction?
# Depends on: cell 2 (variant_df). "cos" = cosine(modified, original) of the global 768-d embedding, shown alongside
# so the size of the embedding change is visible next to the size of the prediction change.
# ============================================================================
def summarize(g):
    return pd.Series({
        "n_patients": g["patient_idx"].nunique(), "n_removals": len(g),
        "median_dP_true": g["delta_true"].median(), "min_dP_true": g["delta_true"].min(), "max_dP_true": g["delta_true"].max(),
        "median_|dP_true|": g["delta_true"].abs().median(),
        "argmax_flip_rate": g["flip"].mean(),
        "median_cos_to_original": g["cos"].median(),
        "window_avail_changed": g["avail_changed"].mean(),
    })

print("=== By removal mode and kind ===")
print(variant_df.groupby(["mode", "kind"]).apply(summarize).round(4).to_string())

print("\n=== Cumulative removal, by number of top drivers removed (k) ===")
print(variant_df[variant_df["kind"] == "cumulative"].groupby(["mode", "label"]).apply(summarize)
      [["n_patients", "median_dP_true", "median_|dP_true|", "argmax_flip_rate", "median_cos_to_original", "window_avail_changed"]].round(4).to_string())

In [0]:
# Follow-up checks on variant_df (uses Cell 2's P_base, P_req, pi_v, emb_v, tgt). Aggregates / patient indices only.
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
variant_df["delta_logit"] = lg(P_req[emb_v, tgt]) - lg(P_base[pi_v, tgt])           # change on the logit scale: immune to sigmoid saturation

pb = P_base[:, tgt]
print(f"baseline P({C['target_class']}) across patients: min {pb.min():.3f}, median {np.median(pb):.3f}, max {pb.max():.3f}")
print("\n=== change in logit of the true class ===")
print(variant_df.groupby(["mode", "kind"])["delta_logit"].agg(median="median", min="min", max="max").round(3).to_string())
print("\n=== cumulative removals: units removed and embedding change by k ===")
print(variant_df[variant_df["kind"] == "cumulative"].groupby(["mode", "label"]).agg(
    median_units=("n_units", "median"), median_cos=("cos", "median")).round(3).to_string())
print("\n=== which patients carry the largest |dP| (patient index) ===")
print(variant_df.groupby("patient_idx")["delta_true"].apply(lambda s: s.abs().max()).sort_values(ascending=False).head(5).round(3).to_string())

In [0]:
# ============================================================================
# Exact logit decomposition of the frozen fusion layer for the selected patients (no perturbation, no sampling).
# logit(true class) = bias + sum_j w_j * x_j, with x = z-scored local sims, z-scored global sims, standardized age.
# Uses Cell 1/2 objects: l_base, g_base, age_pt, FUSION_W/B, z-score stats, P_base, proto_class, local_projection_map.
# Prints aggregates and patient-level counts only.
# ============================================================================
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
nl, ng = n_local_prototypes, n_global_prototypes
xl = (l_base - l_mean) / l_std
xg = (g_base - g_mean) / g_std
xa = (age_pt - age_mean) / (age_std + 1e-8)
w = FUSION_W[tgt]                                                                  # weights of the true class's logit
c_l, c_g, c_a = w[:nl] * xl, w[nl:nl + ng] * xg, w[-1] * xa                        # each input's contribution to that logit
logit = c_l.sum(1) + c_g.sum(1) + c_a + FUSION_B[tgt]
assert np.allclose(logit, lg(P_base[:, tgt]), atol=0.05), "decomposition does not reproduce the fused logit"

loc_class = np.array([local_projection_map[i]["class_name"] for i in range(nl)])
glob_class = np.array([proto_class[i] for i in range(ng)])
is_l, is_g = loc_class == C["target_class"], glob_class == C["target_class"]
tab = pd.DataFrame({
    "bias": np.full(len(pts), FUSION_B[tgt]), "age": c_a,
    f"local, {C['target_class']} prototypes": c_l[:, is_l].sum(1), "local, other-class prototypes": c_l[:, ~is_l].sum(1),
    f"global, {C['target_class']} prototypes": c_g[:, is_g].sum(1), "global, other-class prototypes": c_g[:, ~is_g].sum(1)})
tab["total_logit"] = tab.sum(axis=1)
print("=== contribution to the true-class logit, across patients (logit units) ===")
print(tab.quantile([0, 0.5, 1]).T.rename(columns={0: "min", 0.5: "median", 1: "max"}).round(2).to_string())

prox = select_best_patients(C["target_class"], n_patients=C["n_patients"])["prototype_idx"].to_numpy()
c_prox = c_g[np.arange(len(pts)), [global_proto_indices.index(p) for p in prox]]   # the proximal prototype LOO tracked
print(f"\nproximal global prototype (the one LOO tracked): median contribution {np.median(c_prox):.2f} logits "
      f"vs median total logit {np.median(logit):.2f}")

top = np.concatenate([c_l, c_g], axis=1).argmax(axis=1)                            # single largest positive contributor per patient
who = pd.Series([("local " if t < nl else "global ") + (loc_class[t] if t < nl else glob_class[t - nl]) for t in top])
print("\nbranch / class of each patient's largest single contributor:")
print(who.value_counts().to_string())

In [0]:
# ============================================================================
# Which part of the logit moves when the drivers are removed? Delta-logit = sum_j w_j * (x_j[removed] - x_j[baseline]) exactly,
# so split it by input group. Run after the decomposition cell (uses xl, xg, w, nl, ng, is_g, prox, c_g from it).
# ============================================================================
prox_pos = np.array([global_proto_indices.index(p) for p in prox])
xg_req = (g_req - g_mean) / g_std
xl_req = (l_req - l_mean) / l_std
dxg, dxl = xg_req[emb_v] - xg[pi_v], xl_req[emb_v] - xl[pi_v]
wg, wl = w[nl:nl + ng], w[:nl]
rows = np.arange(len(emb_v))
d_prox = wg[prox_pos[pi_v]] * dxg[rows, prox_pos[pi_v]]                            # the proximal global prototype LOO tracked
d_g_target = (dxg[:, is_g] * wg[is_g]).sum(1) - d_prox                             # the other global GDM prototypes
d_g_other = (dxg[:, ~is_g] * wg[~is_g]).sum(1)                                     # global prototypes of the other classes
d_local = (dxl * wl).sum(1)                                                        # whole local branch
check = lg(P_req[emb_v, tgt]) - lg(P_base[pi_v, tgt])
assert np.allclose(d_prox + d_g_target + d_g_other + d_local, check, atol=0.05), "delta decomposition does not add up (age is held fixed)"
dd = pd.DataFrame({"mode": variant_df["mode"].values, "kind": variant_df["kind"].values, "proximal prototype": d_prox,
                   "other global GDM prototypes": d_g_target, "global other-class prototypes": d_g_other,
                   "local branch": d_local, "total": check})
print("=== median change in each part of the true-class logit, by removal type ===")
print(dd.groupby(["mode", "kind"]).median().round(3).to_string())

In [0]:
# Do any removals push P(true class) from >= 0.5 to < 0.5? (the cutoff your metric suite uses)
crossed = (P_base[pi_v, tgt] >= 0.5) & (P_req[emb_v, tgt] < 0.5)
print(f"removals crossing below 0.5: {int(crossed.sum())} of {len(crossed)} | patients affected: {len(set(pi_v[crossed]))}")

In [0]:
# ============================================================================
# CELL 4 -- Patients across confidence bands: select, re-embed baselines, per-patient reproduction gate.
# Depends on: cells 1-2 (frozen pipeline + helpers: embed_all, extract_windows, local_pooled, tri_window_ends,
# TRI_NAMES, embed_variant's logic, _meds_ds, ...), plus payload, Y_fusion_test, p_test, pids_test_str,
# proto_class, global_proto_indices, tgt.
#
# Population: TRUE-GDM test patients, banded by the frozen fusion model's own baseline P(GDM) (stored p_test).
# The top band (>= 0.99) is where the original 15 patients sat, so it doubles as the comparison group.
# ============================================================================
C.update({
    "conf_bin_edges": [0.0, 0.5, 0.9, 0.99, 1.0 + 1e-9],
    "conf_bin_labels": ["<0.5", "0.5-0.9", "0.9-0.99", ">=0.99"],
    "n_per_band": 25,                # random sample per band (all of them if a band has fewer)
    "verify_global_cos": 0.999,      # per-patient: re-embedded global embedding vs stored (it was 1.00000 with the extraction-style loader)
    "max_gate_fail_frac": 0.10,      # patients failing the gate are EXCLUDED and counted; stop if more than this share fail
})
labels = C["conf_bin_labels"]

# ---- select true-GDM test patients per band ----
is_t = np.asarray(Y_fusion_test)[:, tgt] == 1
p_t = np.asarray(p_test)[:, tgt]
band_all = np.asarray(pd.cut(p_t, bins=C["conf_bin_edges"], labels=labels, right=False).astype(str))
rng_sel = np.random.RandomState(C["seed"])
flow = pd.DataFrame(0, index=labels, columns=["true_gdm_in_test", "selected", "failed_gate", "no_driver_present", "analyzed"])
sel_idx = []
for lab in labels:
    idx = np.flatnonzero(is_t & (band_all == lab))
    take = rng_sel.choice(idx, min(C["n_per_band"], len(idx)), replace=False) if len(idx) else np.array([], dtype=int)
    sel_idx += take.tolist()
    flow.loc[lab, ["true_gdm_in_test", "selected"]] = [len(idx), len(take)]
pts_x = [pids_test_str[i] for i in sel_idx]
band_x = band_all[sel_idx]
age_x = np.asarray(payload["age_test"], dtype=np.float32)[sel_idx]
print(flow[["true_gdm_in_test", "selected"]].T.to_string())

# ---- events for ALL selected patients in one pyarrow read, built exactly as the extraction builds them ----
def events_like_extraction_many(pids):
    df = _meds_ds.to_table(columns=["subject_id", "time", "code"], filter=pc.field("subject_id").isin([int(p) for p in pids])).to_pandas()
    df = df.sort_values(["subject_id", "time"])
    rows = zip(df["subject_id"].to_numpy().tolist(), df["time"].to_numpy().astype("datetime64[us]").tolist(), df["code"].to_numpy().tolist())
    out = {}
    for sid, prows in groupby(rows, key=itemgetter(0)):                                   # same nested groupby as the extraction
        out[sid] = [{"time": t, "measurements": [{"code": c} for _, _, c in trows]} for t, trows in groupby(prows, key=itemgetter(1))]
    return out

_ev = events_like_extraction_many(pts_x)
assert all(int(p) in _ev for p in pts_x), "a selected patient has no events"
events_x = [_ev[int(p)] for p in pts_x]

# ---- window anchors ----
fx = (spark.table(C["flags_path"]).select("person_id", "recorded_episode_start", "inferred_episode_start")
      .filter(sf.col("person_id").cast("string").isin(pts_x)).toPandas())
fx["person_id"] = fx["person_id"].astype(str)
fx["episode_start"] = pd.concat([pd.to_datetime(fx["recorded_episode_start"]), pd.to_datetime(fx["inferred_episode_start"])], axis=1).min(axis=1, skipna=True)
ep_x = fx.set_index("person_id")["episode_start"]
assert all(p in ep_x.index and pd.notna(ep_x[p]) for p in pts_x), "missing episode_start for a selected patient"
W_ENDS_x = np.stack([tri_window_ends(ep_x[p]) for p in pts_x])


def embed_variant_x(pid, events, wends):
    times, reps = embed_all(pid, events)
    g = reps[np.flatnonzero(times == times.max())[0]]                                    # last event (first of any ties), as in new_v5
    w, av = extract_windows(times, reps, wends)
    return g, w, av


# ---- baselines: re-embed every selected patient's UNMODIFIED sequence ----
print(f"re-embedding {len(pts_x)} baselines ...")
base_x = []
for i, (p, ev) in enumerate(zip(pts_x, events_x)):
    base_x.append(embed_variant_x(p, ev, W_ENDS_x[i]))
    if (i + 1) % 25 == 0:
        print(f"  {i + 1}/{len(pts_x)}")
E_base_x = np.stack([b[0] for b in base_x]); W_base_x = np.stack([b[1] for b in base_x]); AV_base_x = np.stack([b[2] for b in base_x])
l_base_x, D_base_x, mask_base_x = local_pooled(W_base_x, AV_base_x)
g_base_x = sims_from_embeddings(global_adapter, global_proto_layer, E_base_x)

# ---- per-patient gate: global cosine, local window set + delta rows, pooled local similarities vs the stored tables ----
n_x = len(pts_x)
cos_glob_x = (unit(E_base_x) * unit(np.stack([global_X_test[global_pid_to_row[p]] for p in pts_x]))).sum(1)
st_x = (spark.table(C["local_emb_table"]).filter(sf.col("patient_id").cast("string").isin(pts_x))
        .filter(sf.col("unavailable") == 0).toPandas())
st_x["patient_id"] = st_x["patient_id"].astype(str)
st_x = st_x[~st_x[emb_cols].isna().any(axis=1)]
v_i, w_i = np.nonzero(mask_base_x)
mine_df = pd.DataFrame({"pi": v_i, "win": [TRI_NAMES[w] for w in w_i]})
mine_sets = mine_df.groupby("pi")["win"].apply(frozenset)
st_sets = st_x.groupby("patient_id")["window_name"].apply(frozenset)
win_ok = np.array([mine_sets.get(i, frozenset()) == st_sets.get(pts_x[i], frozenset()) for i in range(n_x)])
st_emb = st_x.set_index(["patient_id", "window_name"])[emb_cols]
mine_keys = pd.MultiIndex.from_arrays([[pts_x[i] for i in v_i], [TRI_NAMES[w] for w in w_i]])
ref_x = st_emb.reindex(mine_keys).to_numpy(dtype=np.float32)                              # NaN where a window is missing from the stored table
rel_x = np.linalg.norm(D_base_x[v_i, w_i] - ref_x, axis=1) / np.nanmedian(np.linalg.norm(ref_x, axis=1))
rel_max = pd.Series(rel_x).groupby(v_i).max().reindex(range(n_x)).to_numpy()              # NaN if a patient has no rows / missing windows
sim_err_x = np.abs(l_base_x - payload["local_similarity_df"].loc[pts_x].to_numpy()).max(axis=1)
gate_ok = (win_ok & (cos_glob_x >= C["verify_global_cos"]) & (np.nan_to_num(rel_max, nan=np.inf) < C["verify_window_rel_tol"])
           & (sim_err_x < C["verify_local_sim_tol"]))
fail_reason = {"window set": int((~win_ok).sum()), "global cosine": int((cos_glob_x < C["verify_global_cos"]).sum()),
               "delta rows": int((np.nan_to_num(rel_max, nan=np.inf) >= C["verify_window_rel_tol"]).sum()),
               "pooled local sims": int((sim_err_x >= C["verify_local_sim_tol"]).sum())}
for lab in labels:
    flow.loc[lab, "failed_gate"] = int(((band_x == lab) & ~gate_ok).sum())
print(f"gate: {int(gate_ok.sum())}/{n_x} patients reproduce the stored pipeline | failures by check (can overlap): {fail_reason}")
print(f"  global cosine min {cos_glob_x.min():.5f} | max delta-row rel. error {np.nanmax(rel_max):.3e} | max pooled-sim error {sim_err_x.max():.3e}")
assert (~gate_ok).mean() <= C["max_gate_fail_frac"], "too many patients fail the reproduction gate -- STOP; inspect the failure counts above."

# ---- keep only gate-passing patients; everything below is indexed 0..n_ok-1 ----
ki = np.flatnonzero(gate_ok)
pts_x = [pts_x[i] for i in ki]; band_x = band_x[ki]; age_x = age_x[ki]; events_x = [events_x[i] for i in ki]
W_ENDS_x = W_ENDS_x[ki]; E_base_x = E_base_x[ki]; W_base_x = W_base_x[ki]; AV_base_x = AV_base_x[ki]
l_base_x = l_base_x[ki]; g_base_x = g_base_x[ki]
P_base_x = fused_probs(l_base_x, g_base_x, age_x)

# proximal prototype = most similar same-class GLOBAL prototype (same definition as select_best_patients, from the stored sims)
cls_proto = [p for p in global_proto_indices if proto_class[p] == C["target_class"]]
prox_idx = payload["global_similarity_df"].loc[pts_x, cls_proto].idxmax(axis=1).to_numpy()
prox_pos_x = np.array([global_proto_indices.index(p) for p in prox_idx])
print(f"re-embedded baseline vs stored p_test: max |dP| = "
      f"{np.abs(P_base_x[:, tgt] - np.asarray(p_test)[[test_row[p] for p in pts_x], tgt]).max():.4f}")

In [0]:
# ============================================================================
# CELL 5 -- Remove the LOO-identified drivers for every analyzed patient; both branches + frozen fusion; logit decomposition.
# Depends on: cell 4 and cells 1-2 helpers (units_for_code, apply_removal, local_pooled, ...).
# Same removal definitions as before: mode "event" (whole events containing the code) / "code" (only that code's measurement);
# "single" (one driver at a time) / "cumulative" (top-k drivers in LOO order). The drivers are the Bonferroni-significant codes
# found in the original high-confidence patients, applied unchanged to every band.
# ============================================================================
sig_set_x = set(population_confirmation[C["target_class"]].query("significant_bonferroni")["code"])
ranked_x = [c for c in loo_results[C["target_class"]]["mean_proximal_sim_drop"].sort_values(ascending=False).index if c in sig_set_x]

requests_x, key2idx_x = [], {}
def request_x(pi, mode, units):
    key = (pi, mode, frozenset(units))
    if key not in key2idx_x:
        key2idx_x[key] = len(requests_x); requests_x.append(key)
    return key2idx_x[key]

specs_x, no_driver = [], np.zeros(len(pts_x), dtype=bool)
for pi, events in enumerate(events_x):
    present = [c for c in ranked_x if any(m["code"] == c for e in events if not is_birth_event(e) for m in e["measurements"])]
    no_driver[pi] = len(present) == 0
    for mode in C["modes"]:
        per_code = {c: set(units_for_code(events, c, mode)) for c in present}
        variants = [("single", r + 1, per_code[c]) for r, c in enumerate(present)]
        variants += [("cumulative", k, set().union(*[per_code[c] for c in present[:k]])) for k in range(1, len(present) + 1)]
        for kind, label, units in variants:
            if not units or not apply_removal(events, units):
                continue
            specs_x.append({"pi": pi, "mode": mode, "kind": kind, "label": label, "k_max": len(present), "n_units": len(units),
                            "emb_idx": request_x(pi, mode, units)})
for lab in labels:
    flow.loc[lab, "no_driver_present"] = int((no_driver & (band_x == lab)).sum())
print(f"{int(no_driver.sum())} of {len(pts_x)} patients have none of the significant drivers and cannot be perturbed")
print(f"{len(specs_x)} driver removals -> {len(requests_x)} unique re-embeddings (each one CLMBR forward pass, serving both branches)")

# ---- embed (the only slow step) ----
G_req_x = np.empty((len(requests_x), E_base_x.shape[1]), dtype=np.float32)
W_req_x = np.empty((len(requests_x),) + W_base_x.shape[1:], dtype=np.float32)
AV_req_x = np.empty((len(requests_x), W_base_x.shape[1]), dtype=bool)
for n, (pi, mode, units) in enumerate(requests_x):
    G_req_x[n], W_req_x[n], AV_req_x[n] = embed_variant_x(pts_x[pi], apply_removal(events_x[pi], units), W_ENDS_x[pi])
    if (n + 1) % 200 == 0:
        print(f"  embedded {n + 1}/{len(requests_x)}")

# ---- batched: both branches + fusion ----
pi_req = np.array([k[0] for k in requests_x])
g_req_x = sims_from_embeddings(global_adapter, global_proto_layer, G_req_x)
l_req_x, _, _ = local_pooled(W_req_x, AV_req_x)
ok_x = ~np.isnan(l_req_x).any(axis=1)                                                   # variants with no surviving local rows have no fusion input
P_req_x = np.full((len(requests_x), len(CLASS_NAMES)), np.nan, dtype=np.float32)
P_req_x[ok_x] = fused_probs(l_req_x[ok_x], g_req_x[ok_x], age_x[pi_req[ok_x]])
cos_req_x = (unit(G_req_x) * unit(E_base_x)[pi_req]).sum(axis=1)

keep_x = [s for s in specs_x if ok_x[s["emb_idx"]]]
pi_v = np.array([s["pi"] for s in keep_x]); emb_v = np.array([s["emb_idx"] for s in keep_x])
print(f"{len(specs_x) - len(keep_x)} removals dropped (no local window rows left)")

# ---- exact logit decomposition of the change (age is held fixed, so it contributes nothing) ----
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
nl, ng = n_local_prototypes, n_global_prototypes
w_t = FUSION_W[tgt]; wl, wg = w_t[:nl], w_t[nl:nl + ng]
glob_class = np.array([proto_class[i] for i in range(ng)]); is_g = glob_class == C["target_class"]
xg_b, xl_b = (g_base_x - g_mean) / g_std, (l_base_x - l_mean) / l_std
dxg = (g_req_x[emb_v] - g_mean) / g_std - xg_b[pi_v]
dxl = (l_req_x[emb_v] - l_mean) / l_std - xl_b[pi_v]
rows = np.arange(len(emb_v)); pp_v = prox_pos_x[pi_v]
d_prox = wg[pp_v] * dxg[rows, pp_v]
d_g_target = (dxg[:, is_g] * wg[is_g]).sum(1) - d_prox
d_g_other = (dxg[:, ~is_g] * wg[~is_g]).sum(1)
d_local = (dxl * wl).sum(1)
base_logit = lg(P_base_x[pi_v, tgt]); d_logit = lg(P_req_x[emb_v, tgt]) - base_logit
assert np.allclose(d_prox + d_g_target + d_g_other + d_local, d_logit, atol=0.05), "delta-logit decomposition does not add up"
base_prox = (wg[prox_pos_x] * xg_b[np.arange(len(pts_x)), prox_pos_x])                  # baseline contribution of the proximal prototype

base_P_v = P_base_x[pi_v, tgt]
variant_x = pd.DataFrame({
    "patient_idx": pi_v, "band": band_x[pi_v], "mode": [s["mode"] for s in keep_x], "kind": [s["kind"] for s in keep_x],
    "label": [s["label"] for s in keep_x], "k_max": [s["k_max"] for s in keep_x], "n_units": [s["n_units"] for s in keep_x],
    "cos": cos_req_x[emb_v], "base_P": base_P_v, "base_logit": base_logit, "base_prox": base_prox[pi_v],
    "delta_true": P_req_x[emb_v, tgt] - base_P_v, "delta_logit": d_logit,
    "flip": P_req_x[emb_v].argmax(axis=1) != P_base_x.argmax(axis=1)[pi_v],
    # crossing below 0.5 is only defined for patients that start at or above it
    "crossed": np.where(base_P_v >= 0.5, (P_req_x[emb_v, tgt] < 0.5).astype(float), np.nan),
    "avail_changed": (AV_req_x[emb_v] != AV_base_x[pi_v]).any(axis=1),
    "d_prox": d_prox, "d_g_target": d_g_target, "d_g_other": d_g_other, "d_local": d_local,
})
for lab in labels:
    flow.loc[lab, "analyzed"] = int(variant_x.loc[variant_x["band"] == lab, "patient_idx"].nunique())
print(f"variant_x: {len(variant_x)} rows over {variant_x['patient_idx'].nunique()} patients")

In [0]:
# ============================================================================
# CELL 6 -- Summaries by confidence band. Depends on: cell 5 (variant_x, flow).
# Unit of analysis = the PATIENT: each patient's removals are collapsed first (so a patient with more driver codes present
# doesn't count more), then summarized across patients in each band. Two removal summaries per patient:
#   "single"          = median over that patient's one-driver-at-a-time removals
#   "cumulative_all"  = removing ALL of that patient's significant drivers together (largest k)
# NOTE: earlier tables pooled removals, so numbers here are not directly comparable to them.
# ============================================================================
vx = variant_x.assign(abs_dP=lambda d: d["delta_true"].abs())
V = pd.concat([vx[vx["kind"] == "single"],
               vx[(vx["kind"] == "cumulative") & (vx["label"] == vx["k_max"])].assign(kind="cumulative_all")])
pl = (V.groupby(["band", "mode", "kind", "patient_idx"])
        .agg(base_P=("base_P", "median"), base_logit=("base_logit", "median"), base_prox=("base_prox", "median"),
             dP=("delta_true", "median"), max_abs_dP=("abs_dP", "max"), dLogit=("delta_logit", "median"), cos=("cos", "median"),
             flip=("flip", "mean"), crossed=("crossed", "mean"), avail=("avail_changed", "mean"),
             d_prox=("d_prox", "median"), d_g_target=("d_g_target", "median"), d_g_other=("d_g_other", "median"), d_local=("d_local", "median"))
        .reset_index())
order = pd.CategoricalDtype(labels, ordered=True)
pl["band"] = pl["band"].astype(order)

print("=== patient flow by baseline-confidence band ===")
print(flow.to_string())

print("\n=== effect of removing the drivers, by band (one value per patient, then summarized across patients) ===")
tab = pl.groupby(["band", "mode", "kind"], observed=True).agg(
    n_patients=("patient_idx", "nunique"), median_base_P=("base_P", "median"), median_base_logit=("base_logit", "median"),
    median_dP=("dP", "median"), max_abs_dP=("max_abs_dP", "max"), median_dLogit=("dLogit", "median"),
    argmax_flip_rate=("flip", "mean"), cross_below_0p5_rate=("crossed", "mean"), median_cos=("cos", "median"),
    window_avail_changed=("avail", "mean"))
print(tab.round(3).to_string())
print("(cross_below_0p5_rate is NaN where no patient in the group starts at >= 0.5)")

print("\n=== where the logit change comes from (median per patient, logit units) ===")
dec = pl.groupby(["band", "mode", "kind"], observed=True).agg(
    baseline_proximal_contribution=("base_prox", "median"), d_proximal=("d_prox", "median"),
    d_other_global_GDM=("d_g_target", "median"), d_global_other_class=("d_g_other", "median"),
    d_local=("d_local", "median"), d_total=("dLogit", "median"))
print(dec.round(3).to_string())

In [0]:
# ============================================================================
# (a) Are the logit changes consistent in direction?  (b) Is each patient's logit concentrated in one or two prototypes or spread
# across many?  (c) Does concentration predict how much removing the drivers moved the logit?   No new forward passes.
# Uses Cell 4-6 objects: xl_b, xg_b, wl, wg, nl, ng, glob_class, band_x, prox_pos_x, pl, labels, C.
# Prints aggregates only (patient indices are not shown).
# ============================================================================
from scipy.stats import spearmanr
TOL = 0.1                                                                          # logit change counted as "no change" if within +/- TOL (arbitrary, for description only)

# (a) direction of the change, one value per patient (median over their single removals, or all drivers removed together)
sc = pl.assign(lower=lambda d: d["dLogit"] < -TOL, higher=lambda d: d["dLogit"] > TOL)
print("=== (a) share of patients whose true-class logit moved down / up by more than 0.1 ===")
print(sc.groupby(["band", "mode", "kind"], observed=True).agg(
    n_patients=("patient_idx", "nunique"), share_lower=("lower", "mean"), share_higher=("higher", "mean")).round(2).to_string())

# (b) concentration of the final logit over prototypes (baseline, before any removal; all gate-passing patients)
loc_class = np.array([local_projection_map[i]["class_name"] for i in range(nl)])
all_class = np.concatenate([loc_class, glob_class]); is_global = np.array([False] * nl + [True] * ng)
c = np.concatenate([wl * xl_b, wg * xg_b], axis=1)                                 # each prototype's contribution to the true-class logit
pos = np.clip(c, 0, None); pos_sum = pos.sum(1)
srt = np.sort(pos, axis=1)[:, ::-1]
n50 = (np.cumsum(srt, axis=1) < 0.5 * pos_sum[:, None]).sum(1) + 1                 # prototypes needed to reach half of the positive contribution
top_i = c.argmax(1); prox_col = nl + prox_pos_x
conc = pd.DataFrame({"band": pd.Categorical(band_x, categories=labels, ordered=True),
                     "top1_logit": srt[:, 0], "top1_share": srt[:, 0] / pos_sum, "top3_share": srt[:, :3].sum(1) / pos_sum,
                     "n_protos_for_half": n50, "prox_share": c[np.arange(len(c)), prox_col] / pos_sum,
                     "top_is_proximal": top_i == prox_col, "top_is_global": is_global[top_i], "top_is_GDM_class": all_class[top_i] == C["target_class"]})
print("\n=== (b) how concentrated is the logit? (positive contributions; per patient, then median across patients in the band) ===")
print(conc.groupby("band", observed=True).agg(
    n=("top1_logit", "size"), top1_logit=("top1_logit", "median"), top1_share=("top1_share", "median"), top3_share=("top3_share", "median"),
    protos_for_half=("n_protos_for_half", "median"), proximal_share=("prox_share", "median"),
    top_is_proximal=("top_is_proximal", "mean"), top_is_global=("top_is_global", "mean"), top_is_GDM_class=("top_is_GDM_class", "mean")).round(2).to_string())

# (c) link to the removal experiment: does a larger baseline proximal share go with a larger drop under removal?
print("\n=== (c) baseline proximal share vs change in logit after removing the drivers (patients with drivers present) ===")
for mode in C["modes"]:
    lk = pl[(pl["mode"] == mode) & (pl["kind"] == "cumulative_all")].set_index("patient_idx").join(conc[["prox_share", "top_is_proximal"]])
    rho = spearmanr(lk["prox_share"], lk["dLogit"], nan_policy="omit")
    med = lk.groupby("top_is_proximal")["dLogit"].agg(n="size", median_dLogit="median").round(3)
    print(f"mode={mode}: Spearman rho(proximal share, dLogit) = {rho.statistic:.2f} (n={len(lk)}, p={rho.pvalue:.2f})")
    print(med.rename(index={True: "top contributor IS the proximal prototype", False: "top contributor is another prototype"}).to_string())

In [0]:
# ============================================================================
# Does the fusion readout respect prototype class structure? Weights of the true-class (GDM) logit by prototype class and branch,
# and the weight sitting on each patient's proximal prototype. Reads weights only; no forward passes, no patient data printed.
# Uses Cell 4-6 objects: FUSION_W, tgt, nl, ng, wl, wg, glob_class, prox_pos_x, C, CLASS_NAMES, local_projection_map.
# ============================================================================
loc_class = np.array([local_projection_map[i]["class_name"] for i in range(nl)])
w_row = FUSION_W[tgt]; thr = 0.05 * np.abs(w_row).max()                              # "near zero" = under 5% of the largest weight in this logit's row (arbitrary)
rows = []
for branch, w_b, cls in [("local", wl, loc_class), ("global", wg, glob_class)]:
    for cn in CLASS_NAMES:
        m = cls == cn
        if m.any():
            rows.append({"branch": branch, "prototype class": cn, "n_prototypes": int(m.sum()), "mean weight": w_b[m].mean(),
                         "min": w_b[m].min(), "max": w_b[m].max(), "near zero": int((np.abs(w_b[m]) < thr).sum())})
print(f"=== fusion weights of the {C['target_class']} logit by prototype class (near zero = |w| < {thr:.3f}) ===")
print(pd.DataFrame(rows).round(3).to_string(index=False))

gdm_g = np.flatnonzero(glob_class == C["target_class"])                              # global GDM-class prototypes
rank_of_prox = np.array([(np.abs(wg[gdm_g]) > abs(wg[p])).sum() + 1 for p in prox_pos_x])   # 1 = largest |weight| among global GDM prototypes
print(f"\nproximal prototypes (one per patient): {len(np.unique(prox_pos_x))} distinct prototypes | "
      f"patients whose proximal prototype has a near-zero weight: {np.mean(np.abs(wg[prox_pos_x]) < thr):.0%} | "
      f"median weight rank among the {len(gdm_g)} global GDM prototypes: {np.median(rank_of_prox):.0f}")